In [1]:
import numpy as np

def monte_carlo_mcc_impact(total_samples: int, iterations: int = 1000000) -> dict:

    probs = np.random.dirichlet( (4, 4, 1, 1), size=iterations)

    matrices = np.array([np.random.multinomial(total_samples, p) for p in probs])
    tp = matrices[:, 0].astype(float)
    tn = matrices[:, 1].astype(float)
    fp = matrices[:, 2].astype(float)
    fn = matrices[:, 3].astype(float)

    def get_mcc_vectorized(t_p, t_n, f_p, f_n):
        num = (t_p * t_n) - (f_p * f_n)
        den = np.sqrt((t_p + f_p) * (t_p + f_n) * (t_n + f_p) * (t_n + f_n))
        return np.divide(num, den, out=np.zeros_like(num, dtype=float), where=den!=0)

    current_mcc = get_mcc_vectorized(tp, tn, fp, fn)

    diff_fn_to_tp = np.where(fn >= 1, get_mcc_vectorized(tp + 1, tn, fp, np.maximum(0, fn - 1)) - current_mcc, np.nan)
    diff_fp_to_tn = np.where(fp >= 1, get_mcc_vectorized(tp, tn + 1, np.maximum(0, fp - 1), fn) - current_mcc, np.nan)
    diff_tp_to_fn = np.where(tp >= 1, get_mcc_vectorized(np.maximum(0, tp - 1), tn, fp, fn + 1) - current_mcc, np.nan)
    diff_tn_to_fp = np.where(tn >= 1, get_mcc_vectorized(tp, np.maximum(0, tn - 1), fp + 1, fn) - current_mcc, np.nan)

    impacts = {
        "fix_false_negative_mean": np.nanmean(diff_fn_to_tp),
        "fix_false_positive_mean": np.nanmean(diff_fp_to_tn),
        "new_false_negative_mean": np.nanmean(diff_tp_to_fn),
        "new_false_positive_mean": np.nanmean(diff_tn_to_fp)
    }
    return impacts

monte_carlo_mcc_impact(round(318/5))

{'fix_false_negative_mean': np.float64(0.034986077428199605),
 'fix_false_positive_mean': np.float64(0.035011087867849844),
 'new_false_negative_mean': np.float64(-0.035541272404374664),
 'new_false_positive_mean': np.float64(-0.03557165804031967)}